# Hamann 2015：密度流、早期下沉羽流与矿物牛眼

In [ ]:
import json
from pathlib import Path

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import RectBivariateSpline

DAYS_PER_YEAR = 365.25

# 可在这里调整要绘制的时刻。
EARLY_YEARS = [1.0, 20.0, 40.0, 70.0]
LATE_YEARS = [1000.0, 2000.0, 3000.0, 4000.0, 5000.0, 6000.0]
NX_FINE, NZ_FINE = 700, 220


def locate_case_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "cases" / "zhengfa"]
    for candidate in candidates:
        if (candidate / "output" / "results.npy").exists():
            return candidate
    raise FileNotFoundError(
        "请先运行 cases/zhengfa/run.py，并从仓库根目录或案例目录执行本 notebook。"
    )


CASE_DIR = locate_case_dir()
OUTPUT_DIR = CASE_DIR / "output"
SIM_DIR = CASE_DIR / "simulation"

mpl.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "font.family": "DejaVu Sans",
    }
)
print(f"Case directory: {CASE_DIR}")

In [ ]:
metadata = json.loads((OUTPUT_DIR / "model_metadata.json").read_text(encoding="utf-8"))
results = np.load(OUTPUT_DIR / "results.npy")
years = np.load(OUTPUT_DIR / "result_times_years.npy")
headings = (OUTPUT_DIR / "results_headings.txt").read_text(encoding="utf-8").splitlines()
delr = np.load(OUTPUT_DIR / "grid_delr_m.npy")
delv = np.load(OUTPUT_DIR / "grid_delv_m.npy")

nlay, ncol = int(metadata["nlay"]), int(metadata["ncol"])
x = np.cumsum(delr) - 0.5 * delr
z = 10.0 - (np.cumsum(delv) - 0.5 * delv)
x_fine = np.linspace(0.0, 100.0, NX_FINE)
z_fine = np.linspace(0.0, 10.0, NZ_FINE)
initial_density = results[0, headings.index("RHO")].reshape(nlay, ncol) * 1000.0
rho_reference = float(np.nanmin(initial_density))

if results.shape != (len(years), len(headings), nlay * ncol):
    raise ValueError(f"结果形状不一致: {results.shape}")
if not np.isfinite(results).all():
    raise ValueError("结果包含 NaN 或 infinity")

budget = flopy.utils.CellBudgetFile(str(SIM_DIR / "gwf_model.bud"), precision="double")
print("Saved years:", years.tolist())
print(f"Grid: {nlay} layers × {ncol} columns; cells={nlay * ncol}")
print(f"Initial density range: {initial_density.min():.3f}–{initial_density.max():.3f} kg/m³")

## 绘图函数

非均匀 MODFLOW 网格上的密度和比流量会被样条插值到规则细网格。密度插值结果被限制在原始单元最小值和最大值之间，避免样条过冲产生虚假的极值。

In [ ]:
def year_index(year):
    matches = np.flatnonzero(np.isclose(years, year))
    if matches.size != 1:
        raise ValueError(f"{year:g} 年不在已保存时刻 {years.tolist()} 中")
    return int(matches[0])


def interpolate_field(field, *, clip=False):
    # RectBivariateSpline 要求坐标递增，因此反转由上至下排列的 z。
    spline = RectBivariateSpline(z[::-1], x, field[::-1], kx=2, ky=3, s=0.0)
    fine = spline(z_fine, x_fine)
    if clip:
        fine = np.clip(fine, np.nanmin(field), np.nanmax(field))
    return fine


def fields_at_year(year):
    idx = year_index(year)
    rho = results[idx, headings.index("RHO")].reshape(nlay, ncol) * 1000.0
    spdis = budget.get_data(text="DATA-SPDIS", totim=float(year * DAYS_PER_YEAR))[0]
    qx = np.asarray(spdis["qx"]).reshape(nlay, ncol)
    qz = np.asarray(spdis["qz"]).reshape(nlay, ncol)
    return rho, qx, qz


def add_flow_overlay(ax, qx, qz):
    qx_fine = interpolate_field(qx)
    qz_fine = interpolate_field(qz)
    speed = np.hypot(qx_fine, qz_fine)
    positive = speed[speed > 0.0]
    scale = np.nanpercentile(positive, 90) if positive.size else 1.0
    linewidth = 0.35 + 1.15 * np.clip(speed / max(scale, 1e-30), 0.0, 1.0)
    ax.streamplot(
        x_fine,
        z_fine,
        qx_fine,
        qz_fine,
        color="white",
        linewidth=linewidth,
        density=(1.45, 0.9),
        arrowsize=0.75,
        arrowstyle="-|>",
        minlength=0.08,
        integration_direction="both",
        broken_streamlines=True,
        zorder=4,
    )
    return float(np.nanmax(speed))


def style_cross_section(ax, year):
    ax.axvline(50.0, color="white", lw=1.0, ls="--", alpha=0.95, zorder=5)
    ax.text(49.2, 9.55, "recharge", color="white", ha="right", va="top", fontsize=8)
    ax.text(50.8, 9.55, "evaporation", color="white", ha="left", va="top", fontsize=8)
    ax.set_xlim(0.0, 100.0)
    ax.set_ylim(0.0, 10.0)
    ax.set_xlabel("Distance x (m)")
    ax.set_ylabel("Elevation z (m)")
    label = "year" if np.isclose(year, 1.0) else "years"
    ax.set_title(f"{year:g} {label}", loc="left", fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)


def density_panel(ax, year, *, vmin=None, vmax=None, levels=32):
    rho, qx, qz = fields_at_year(year)
    rho_fine = interpolate_field(rho, clip=True)
    if vmin is None:
        vmin = float(np.nanmin(rho))
    if vmax is None:
        vmax = float(np.nanmax(rho))
    if np.isclose(vmin, vmax):
        vmax = vmin + 0.01
    contour = ax.contourf(
        x_fine,
        z_fine,
        rho_fine,
        levels=np.linspace(vmin, vmax, levels),
        cmap="turbo",
        extend="both",
    )
    # 少量等密度线帮助辨认羽流边界。
    line_levels = np.linspace(vmin, vmax, 7)[1:-1]
    ax.contour(
        x_fine, z_fine, rho_fine, levels=line_levels, colors="k", linewidths=0.28, alpha=0.35
    )
    max_speed = add_flow_overlay(ax, qx, qz)
    style_cross_section(ax, year)
    ax.text(
        0.012,
        0.965,
        f"ρ = {rho.min():.2f}–{rho.max():.2f} kg m⁻³\n|max q| = {max_speed:.2e} m d⁻¹",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8,
        color="white",
        bbox=dict(facecolor="black", alpha=0.42, edgecolor="none", pad=2.5),
        zorder=6,
    )
    return contour, rho


def save_and_show(fig, filename):
    path = OUTPUT_DIR / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print(f"Wrote {path}")
    plt.show()

## 早期不稳定下沉羽流

这里故意为每个时刻使用独立色标。1 年与 70 年的密度振幅相差很大，若使用 6000 年统一色标，早期羽流会几乎不可见。每个色标都标出了绝对密度，因此不会混淆数值范围。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15.2, 6.6), constrained_layout=True)
for ax, year in zip(axes.flat, EARLY_YEARS, strict=False):
    contour, rho = density_panel(ax, year)
    cbar = fig.colorbar(contour, ax=ax, pad=0.015, fraction=0.035)
    cbar.set_label("Density (kg m⁻³)")
fig.suptitle(
    "Early density instability: descending brine plumes",
    fontsize=15,
    fontweight="bold",
)
save_and_show(fig, "density_flow_early_plumes.png")

## 长期密度流演化

1000–6000 年所有面板使用统一色标，展示下沉高密度盐水、深部侧向流动以及最终环流结构。

In [ ]:
late_density = [fields_at_year(year)[0] for year in LATE_YEARS]
late_vmin = float(min(np.nanmin(field) for field in late_density))
late_vmax = float(max(np.nanmax(field) for field in late_density))

fig, axes = plt.subplots(3, 2, figsize=(15.2, 9.2), constrained_layout=True)
last_contour = None
for ax, year in zip(axes.flat, LATE_YEARS, strict=False):
    last_contour, _ = density_panel(ax, year, vmin=late_vmin, vmax=late_vmax, levels=42)
cbar = fig.colorbar(last_contour, ax=axes, pad=0.012, fraction=0.024, shrink=0.92)
cbar.set_label("Density (kg m⁻³)")
fig.suptitle(
    "Long-term density-driven circulation",
    fontsize=15,
    fontweight="bold",
)
save_and_show(fig, "density_flow_long_term.png")

## 地表矿物牛眼模式

In [ ]:
minerals = ("Calcite", "Gypsum", "Halite")
colors = plt.cm.viridis(np.linspace(0.04, 0.96, len(years)))
fig, axes = plt.subplots(3, 1, figsize=(11.5, 8.5), sharex=True, constrained_layout=True)

for ax, mineral in zip(axes, minerals, strict=False):
    mineral_idx = headings.index(mineral)
    for time_idx, (year, color) in enumerate(zip(years, colors, strict=False)):
        surface = results[time_idx, mineral_idx].reshape(nlay, ncol)[0]
        ax.plot(x, surface, color=color, lw=1.35, label=f"{year:g} y")
    ax.axvline(50.0, color="0.25", lw=0.9, ls="--")
    ax.set_ylabel(f"{mineral}\n(mol L⁻¹ bulk)")
    ax.grid(alpha=0.22)
    ax.set_xlim(0.0, 100.0)
axes[0].legend(ncol=6, fontsize=8, loc="upper left")
axes[-1].set_xlabel("Distance x (m)")
fig.suptitle("Surface mineral precipitation: bull's-eye transect", fontsize=14, fontweight="bold")
save_and_show(fig, "bullseye_surface.png")